# 0) Enviroment

### Imports and Constants

In [116]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import ElasticNet, QuantileRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from mord import LogisticAT

DATA_PATH = "../data/compas-scores-two-years.csv"
_RS = 42
_TARGET = "decile_score"
_TEST_PERCENTAGE = 0.2


### Import data and isolate target

In [ ]:
df = pd.read_csv(DATA_PATH)

y = df.pop(_TARGET)
X = df

### Helper functions


In [ ]:
def get_regression_metrics(model, X_test, y_test, X_train, y_train):
    """
    Calculates and prints key regression performance metrics (MAE, MSE, RMSE and R2).
    
    Parameters:
    model: A fitted sklearn estimator or Pipeline.
    X_test: Test features (DataFrame or Array).
    y_test: True target values.
    X_train: Train features
    y_train: Train target values.
    

    Outputs:
    mae: Mean average error
    mse: Mean square error    
    rmse: Root mean square error
    r2: R2 score
    tr2: Training R2

    
    Contribution: Co-developed by Gemini Pro 3.1, mostly me.
    """
    y_pred = model.predict(X_test)

    y_train_pred = model_elastic_net.predict(X_train)
    tr2 = r2_score(y_train, y_train_pred)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    return mae, mse, rmse, r2, tr2

def print_regression_metrics(test_mae, test_mse, test_rmse, test_r2, train_r2):
    print(f"\nTraining R2: {train_r2:.4f}")
    print(f"Test R2:     {test_r2:.4f}")
    print(f"Difference:  {train_r2 - test_r2:.4f}")

    print(f"\nOther metrics:")
    print(f"MAE: {test_mae:.4f}")
    print(f"MSE: {test_mse:.4f}")
    print(f"RMSE: {test_rmse:.4f}")


# These 2 line were given by Gemini, they force paralelism in GridSearch
from functools import partial
GridSearchCV = partial(GridSearchCV, n_jobs=-1)

# 1) Train and Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=_RS, test_size=_TEST_PERCENTAGE, 
    stratify=y, # To ensure we have the same percentage of decile scores in test and train data 
)

# 2) Linear Models

> **AI Acknowledgements** (Gemini):
- I found out about **Quantile** and **Ordinal** regressions while brainstorming with Gemini. They improve the model's ability to handle discrete, ranked scales and outlier-heavy data by focusing on thresholds and medians rather than just the mathematical average.

- The **TransformedTargetRegressor** is a wrapper that transforms the target variable to approximate a normal distribution in cases of skewness. Also sugested by Gemini it is useful in conjunction with the **Quantile** regression model. By stabilizing the variance and **reducing skew**, the transformation ensures the model's 'pinball loss' function is minimized across a more uniform error distribution.  

- Finally the usage of Splines to break the ~0.5 barries was also sugested by Gemini, which denoted that the linear model can be more
flexible when using them.

> **Toughts and observations:**
- The R2 score for these linear models seems to be capped at ~0.5, which leads us to believe that the dataset is more complex
and non purely linear. This is corroborated by the attempt of using **SplineTransform** to improve feature set and improving R2 score, which didn't budge, 
meaning we maxed out the capabilities of linear models for this feature set.

- While using the **TTR** I chose the np.log and np.exp to treat the right skewness of the target variable.

- TODO: calculate acc, precision and recall with rounded values. After training only.

### ElasticNet (Polinomial) 

##### Model pipeline

In [ ]:
model_elastic_net_pipeline = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(), # ElasticNet is very sensitive to feature scale.
    ElasticNet(
        alpha=1.0, 
        l1_ratio=0.5, 
        max_iter=10000,
        tol=0.01
    ),
)

##### Hyper Parameter Optimization

In [ ]:
# The GridSearchCV was indicated by an LLM in order to optimize hyper parameters

param_grid = {
    'elasticnet__alpha': [0.01, 0.1, 1.0], 
    'elasticnet__l1_ratio': [0.5, 0.9, 0.99],
    'polynomialfeatures__degree': [1, 2]
}

model_elastic_net = GridSearchCV(
    model_elastic_net_pipeline, 
    param_grid, 
    cv=5,
    scoring='neg_mean_squared_error', 
    refit=True # Refit with best model at the end
)
model_elastic_net.fit(X_train, y_train)


print(f"Best Params: {model_elastic_net.best_params_}")

##### Baseline metrics

In [ ]:
metrics = get_regression_metrics(model_elastic_net, X_test, y_test, X_train, y_train)
print_regression_metrics(*metrics)

### Quantile Regression

##### Model pipeline

In [ ]:
model_quantile_base_pipeline = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    QuantileRegressor(
        quantile=0.5,
        alpha=1.0,
        solver='highs'
    ),
)

model_quantile_pipeline = TransformedTargetRegressor(
    regressor=model_quantile_base_pipeline,
    func=np.log,
    inverse_func=np.exp
)

##### Hyper Parameter Optimization

In [ ]:
param_grid_quantile = {
    'regressor__quantileregressor__quantile': [0.25, 0.5, 0.75],
    'regressor__quantileregressor__alpha': [0.01, 0.1, 1.0],
    'regressor__polynomialfeatures__degree': [1, 2]
}

model_quantile = GridSearchCV(
    model_quantile_pipeline,
    param_grid_quantile,
    cv=5,
    scoring='neg_mean_squared_error',
    refit=True
)
model_quantile.fit(X_train, y_train)

print(f"Best Params: {model_quantile.best_params_}")

##### Metrics

In [ ]:
metrics_quantile = get_regression_metrics(model_quantile, X_test, y_test, X_train, y_train)
print_regression_metrics(*metrics_quantile)

### Ordinal Regression

##### Model pipeline

In [ ]:
model_ordinal_pipeline = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    LogisticAT(alpha=1.0)
)

##### Hyper Parameter Optimization

In [ ]:
param_grid_ordinal = {
    'logisticat__alpha': [0.01, 0.1, 1.0],
    'polynomialfeatures__degree': [1, 2]
}

model_ordinal = GridSearchCV(
    model_ordinal_pipeline,
    param_grid_ordinal,
    cv=5,
    scoring='neg_mean_squared_error',
    refit=True
)
model_ordinal.fit(X_train, y_train)

print(f"Best Params: {model_ordinal.best_params_}")

##### Metrics

In [ ]:
metrics_ordinal = get_regression_metrics(model_ordinal, X_test, y_test, X_train, y_train)
print_regression_metrics(*metrics_ordinal)

### ElasticNet (Spline + TTR)

##### Model pipeline

In [ ]:
model_elasticnet_spline_base_pipeline = make_pipeline(
    SplineTransformer(n_knots=5, degree=3),
    StandardScaler(),
    ElasticNet(
        alpha=1.0,
        l1_ratio=0.5,
        max_iter=10000,
        tol=0.01
    ),
)

model_elasticnet_spline_pipeline = TransformedTargetRegressor(
    regressor=model_elasticnet_spline_base_pipeline,
    func=np.log,
    inverse_func=np.exp
)

##### Hyper Parameter Optimization

In [ ]:
param_grid_elasticnet_spline = {
    'regressor__splinetransformer__n_knots': [5, 8],
    'regressor__elasticnet__alpha': [0.01, 0.1, 1.0],
    'regressor__elasticnet__l1_ratio': [0.5, 0.9, 0.99]
}

model_elasticnet_spline = GridSearchCV(
    model_elasticnet_spline_pipeline,
    param_grid_elasticnet_spline,
    cv=5,
    scoring='neg_mean_squared_error',
    refit=True
)
model_elasticnet_spline.fit(X_train, y_train)

print(f"Best Params: {model_elasticnet_spline.best_params_}")

##### Metrics

In [ ]:
metrics_elasticnet_spline = get_regression_metrics(model_elasticnet_spline, X_test, y_test, X_train, y_train)
print_regression_metrics(*metrics_elasticnet_spline)

# 3) Non-Linear Models

### Random forest

> Obversations:
- We can't seem to break 0.5 barrier, the literature points out that this is normal for human behaviour variance explainance.
Will have to review with professor

##### Baseline for RF

In [128]:
rf_baseline = RandomForestRegressor(
    n_estimators=1000,
    max_features='sqrt',
    max_depth=10,
    min_samples_leaf=5,
    random_state=_RS,
    n_jobs=-1
)

rf_baseline.fit(X_train, y_train)

metrics_rf_baseline = get_regression_metrics(rf_baseline, X_test, y_test, X_train, y_train)
print_regression_metrics(*metrics_rf_baseline)


Training R2: 0.4851
Test R2:     0.4928
Difference:  -0.0078

Other metrics:
MAE: 1.6425
MSE: 4.1411
RMSE: 2.0350


##### Model pipeline

In [114]:
model_random_forest_pipeline = make_pipeline(
    RandomForestRegressor(
        n_estimators=200,      
        max_features='sqrt',    # random subset of features
        random_state=_RS
    ),
)

##### Hyper parameter optimization

In [130]:
param_grid_random_forest = {
    'randomforestregressor__n_estimators': [100, 200, 300],
    'randomforestregressor__max_depth': [3, 6, 9],
    'randomforestregressor__min_samples_leaf': [1, 2, 4],
}

model_random_forest = GridSearchCV(
    estimator=model_random_forest_pipeline, 
    param_grid=param_grid_random_forest,
    cv=5, 
    scoring='neg_mean_absolute_error',
    refit=True
)

model_random_forest.fit(X_train, y_train)

print(f"Best Params: {model_random_forest.best_params_}")

Best Params: {'randomforestregressor__max_depth': 9, 'randomforestregressor__min_samples_leaf': 2, 'randomforestregressor__n_estimators': 100}


##### Metrics

In [131]:
metrics_random_forest = get_regression_metrics(model_random_forest, X_test, y_test, X_train, y_train)
print_regression_metrics(*metrics_random_forest)


Training R2: 0.4851
Test R2:     0.4862
Difference:  -0.0011

Other metrics:
MAE: 1.6578
MSE: 4.1955
RMSE: 2.0483


##### 